> 在线讲义：[第 50 章 · DiT 与 MM-DiT：用 Transformer 当扩散的主干](https://wikieden.github.io/beyond-transformer/lessons/50-dit-mmdit.html) · 本书《超越 Transformer》

# 第 50 章 · DiT 与 MM-DiT：用 Transformer 当扩散的主干

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/50-dit-mmdit.html) · 仓库里 `beyond-transformer/notes/lessons/50-dit-mmdit.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 token 数 | $(I/p)^2$ | `n_tokens(32, p)` | p = 8、4、2 → 16、64、256 |
| 第二步 adaLN | $\gamma \odot \text{LN}(x) + \beta$ | `ada_ln(x, gamma, beta)` | [−0.612, 0, 0.712]；[−1.637, 0, 1.837] |
| 第三步 门 α | $x + \alpha \odot f$ | `gated_residual(x, f_out, alpha)` | α = 0 → [1, 2, 3]；α = 0.2 → [1.1, 1.9, 3.2] |
| 第五步 联合注意力 | 拼接 | `joint_quadrants(4, 2)` | 6 × 6，四块：4 × 4、4 × 2、2 × 4、2 × 2 |
| 第五步 logit-normal | $\sigma(u)$ | `logit_normal_t([-2, -1, 0, 1, 2])` | 0.119、0.269、0.5、0.731、0.881 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `n_tokens`**：潜空间 I × I 按 p × p 切块：T = (I/p)²。

In [ ]:
def n_tokens(I, p):
    """潜空间 I × I 按 p × p 切块：T = (I/p)²。"""
    return (I // p) ** 2

**函数 `layer_norm`**：普通 LayerNorm：减平均、除标准差（演示用，不加 eps）。

In [ ]:
def layer_norm(x):
    """普通 LayerNorm：减平均、除标准差（演示用，不加 eps）。"""
    x = np.asarray(x, float)
    return (x - x.mean()) / x.std()

**函数 `ada_ln`**：adaLN：γ、β 由条件算出，γ ⊙ LN(x) + β。

In [ ]:
def ada_ln(x, gamma, beta):
    """adaLN：γ、β 由条件算出，γ ⊙ LN(x) + β。"""
    return np.asarray(gamma) * layer_norm(x) + np.asarray(beta)

**函数 `gated_residual`**：adaLN-Zero：x + α ⊙ 子层输出；α = 0 时原样输出。

In [ ]:
def gated_residual(x, f_out, alpha):
    """adaLN-Zero：x + α ⊙ 子层输出；α = 0 时原样输出。"""
    return np.asarray(x, float) + alpha * np.asarray(f_out, float)

**函数 `joint_quadrants`**：MM-DiT 联合注意力的分数表大小，以及四块：图看图、图看文、文看图、文看文。

In [ ]:
def joint_quadrants(n_img, n_txt):
    """MM-DiT 联合注意力的分数表大小，以及四块：图看图、图看文、文看图、文看文。"""
    n = n_img + n_txt
    return {"table": (n, n), "img_img": (n_img, n_img), "img_txt": (n_img, n_txt),
            "txt_img": (n_txt, n_img), "txt_txt": (n_txt, n_txt)}

**函数 `logit_normal_t`**：logit-normal：t = sigmoid(u)。

In [ ]:
def logit_normal_t(u):
    """logit-normal：t = sigmoid(u)。"""
    return 1 / (1 + np.exp(-np.asarray(u, float)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 token 数：潜空间 32 × 32 × 4")
for p in (8, 4, 2):
    print(f"p = {p}：T =", n_tokens(32, p))

In [ ]:
print("\n第二步 adaLN：x = [1, 2, 3]")
x = np.array([1, 2, 3.0])
print("LN(x) =", layer_norm(x))
print("t 很大：", ada_ln(x, [0.5] * 3, [0, 0, 0.1]))
print("t 很小：", ada_ln(x, [1.5] * 3, [0.2, 0, 0]))

In [ ]:
print("\n第三步 adaLN-Zero 的门：子层输出 [0.5, −0.5, 1]")
for a in (0.0, 0.2):
    print(f"α = {a}：", gated_residual(x, [0.5, -0.5, 1], a))

In [ ]:
print("\n第五步 MM-DiT：4 个图像 token + 2 个文字 token")
print(joint_quadrants(4, 2))
print("logit-normal：u = −2..2 → t =", logit_normal_t([-2, -1, 0, 1, 2]))

## 3. 代码里才看得到的

- DiT 官方代码写的是 `x * (1 + scale) + shift`，scale 初始为 0 时缩放就是 1；讲义写成 γ ⊙ LN(x) + β，两者只差一个"+1"的约定。
- MM-DiT 的"各用各的权重"在代码里就是：拼接前图像 token 和文字 token 分别乘自己的 $W^Q, W^K, W^V$，拼接后算一张大分数表，再按位置拆回两边，各进各的 MLP。

## 4. 自己改一改

- `gated_residual` 的 α 从 0 改到 1，看输出怎么从 x 变成 x + f。
- `logit_normal_t` 里把 u 改成从 𝒩(1, 1) 抽（m = 1），看 t 是不是偏向噪声那头。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 画师换成 Transformer：潜图切块当 token，p 减一半算四倍。
> 步数类别进 MLP，算出缩放和平移，调好 LayerNorm 再进层。
> 残差前面加道门，初始关上先旁听。
> 算力越大画越好，XL 斜杠二最强。
> 图文拼成一条队，各用各的权重，一起做注意力；t 用 logit 正态，多练中间段。